# cancer-recon-apoptosis — Step 2: Cancer-restricted target discovery (Colab)

**Goal:** a ranked shortlist of CELL-SURFACE receptors enriched on CANCER cells vs matched NORMAL cells — feeds Step 3 (specificity audit) and Step 4 (reward).

**Engine:** CELLxGENE Census (data) → cancer-vs-normal receptor differential (2b) → surfaceome filter (2b.2) → **LIANA+** communication annotation (2c, next).

**Runtime:** CPU is fine; **High-RAM** helps the fetch. **No GPU.**

**Run log:** every run is teed to `runs/logs/step2_<UTC>_<gitSHA>.log` and auto-downloaded by the last cell — share that file. The SHA links output to the exact code version.

**Order:** 1 clone → 2 install+log → 3 explore → 4 fetch (~30 min, one-time, cached) → 5 shortlist → 5b surfaceome filter → 6 finalize. If data's already fetched, skip 4.

## 1. Clone / pull repo (bootstrap)

In [ ]:
#@title Cell 1 — clone / pull repo (idempotent)
print('[CELL 1] ▶ clone or pull repo')
import os
from pathlib import Path
REPO_URL = 'https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git'
REPO_DIR = Path('/content/cancer-recon-apoptosis')
if REPO_DIR.exists():
    print('  updating to latest (discards local untracked logs)')
    !cd {REPO_DIR} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log --oneline -1
for s in ('scripts/03_census_fetch.py','scripts/04_receptor_targets.py','scripts/05_surfaceome_filter.py','scripts/runlog.py'):
    assert (REPO_DIR / s).exists(), f'missing {s}'
print('[CELL 1] ✓ done')

## 2. Start run log + install deps

In [ ]:
#@title Cell 2 — start run log + install (cellxgene-census / scanpy / liana)
import sys
from scripts.runlog import new_log, run_logged, finalize
RUNLOG = new_log('step2', repo_dir='.')
import importlib.util
need = [m for m in ('cellxgene_census','scanpy','liana','omnipath') if importlib.util.find_spec(m) is None]
if need:
    run_logged([sys.executable,'-m','pip','install','-q','cellxgene-census','scanpy','liana'],
               RUNLOG, label='pip install cellxgene-census scanpy liana')
else:
    print('  deps already installed')
import cellxgene_census, scanpy
print('  cellxgene_census', cellxgene_census.__version__, '| scanpy', scanpy.__version__)
print('[CELL 2] ✓ done')

## 3. EXPLORE — look at the data first

Per tissue: total cells, `disease` values, top `cell_type` values, malignant/neoplastic/epithelial annotation. scripts/04 adapts automatically (lung→malignant, colon→neoplastic, breast→epithelial).

In [ ]:
#@title Cell 3 — explore Census metadata (cheap)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
rc = run_logged([sys.executable,'-u','scripts/03_census_fetch.py','explore'], RUNLOG)
print(f'[CELL 3] exit={rc}', '✓' if rc==0 else '✗')

## 4. FETCH — tumour + normal AnnData (~30 min, one-time, cached/idempotent)

In [ ]:
#@title Cell 4 — fetch tumour + normal cells (~30 min, one-time)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
rc = run_logged([sys.executable,'-u','scripts/03_census_fetch.py','fetch'], RUNLOG)
print(f'[CELL 4] exit={rc}', '✓' if rc==0 else '✗')
!ls -la data/cellxgene 2>/dev/null

## 5. Receptor shortlist — cancer vs normal differential

Adaptive cancer-cell selection → Wilcoxon on the LIANA receptor set → per-tissue → pan-cancer aggregate → `data/cellxgene/targets_shortlist.csv`. Fast.

In [ ]:
#@title Cell 5 — cancer-enriched receptor shortlist
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
rc = run_logged([sys.executable,'-u','scripts/04_receptor_targets.py'], RUNLOG)
print(f'[CELL 5] exit={rc}', '✓' if rc==0 else '✗')

## 5b. Surfaceome filter — keep only ligand-targetable receptors

Intersect the shortlist with the human surfaceome (OmniPath `plasma_membrane_transmembrane`, incl. Bausch-Fluck SURFY; curated fallback). Drops non-surface 'receptors' (FLOT1/RACK1/…). → `data/cellxgene/targets_surface_shortlist.csv` + the actionable set.

In [ ]:
#@title Cell 5b — surfaceome filter (actionable surface targets)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
rc = run_logged([sys.executable,'-u','scripts/05_surfaceome_filter.py'], RUNLOG)
print(f'[CELL 5b] exit={rc}', '✓' if rc==0 else '✗')
import os, pandas as pd
p = 'data/cellxgene/targets_surface_shortlist.csv'
if rc==0 and os.path.exists(p):
    print('\nTop 25 cancer-enriched SURFACE receptors:')
    print(pd.read_csv(p).head(25).to_string(index=False))

## 6. Finalize — download the run log

In [ ]:
#@title Cell 6 — finalize + download run log
from scripts.runlog import new_log, finalize
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
finalize(RUNLOG)
!ls -la runs/logs 2>/dev/null

## 7. Next: pick the target thesis, then Step 2c (LIANA communication)

Share the downloaded run log + `targets_surface_shortlist.csv`. We decide the target thesis (over-expressed-receptor vs DR5-death-receptor vs hybrid), then annotate which shortlisted receptors drive tumour-enriched ligand-receptor communication (LIANA+).